In [7]:
import torch
import math
from torch import Tensor
from typing import Optional, Callable, Iterable
from jaxtyping import Float, Int

# 1. 检测并定义设备 (NVIDIA 设备名为 'cuda')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

# ==========================================
# 优化器定义保持不变（它能自动适应参数所在的设备）
# ==========================================
class SGD(torch.optim.Optimizer):
    def __init__(self, params: Iterable[torch.nn.Parameter], lr: float = 1e-3):
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                if p.grad is None: continue
                state = self.state[p]
                if "t" not in state: state["t"] = 0
                t = state["t"]
                
                # 核心逻辑：如果 p 在 GPU 上，这里的所有运算都会在 GPU 上执行
                p.data -= lr / math.sqrt(t + 1) * p.grad.data
                state["t"] = t + 1
        return loss

# ==========================================
# 2. 搬运到 NV 设备 (CUDA)
# ==========================================

torch.manual_seed(42)

# 创建权重并立刻搬运到 GPU -> .to(device)
weights = torch.nn.Parameter(torch.randn((10, 10), device=device))

# 优化器绑定已经在 GPU 上的权重
optimizer = SGD([weights], lr=1e3)

for epoch in range(10):
    optimizer.zero_grad()
    
    # 所有的计算 (平方、均值) 现在都在 GPU 核心里跑
    loss = (weights**2).mean()
    loss.backward()
    optimizer.step()
    
    print(f"\n第 {epoch + 1} 轮 | 设备: {weights.device} | Loss: {loss.item():.6f}")
    
    # 打印时需要从 GPU 拷回 CPU 才能转 numpy
    curr_weights = weights.detach().cpu().numpy()
    for row in curr_weights:
        print(" ".join([f"{val:8.4f}" for val in row]))

print("\nCUDA 训练演示结束。")

当前使用的设备: cuda

第 1 轮 | 设备: cuda:0 | Loss: 0.855126
 -3.6864 -41.0661   3.2690 -16.1321  36.5636 -12.4067  12.3394  15.5330 -10.0313  24.2316
 31.5804   5.7630   1.7588  -3.7855  21.2882 -35.2955  13.5759 -13.0740 -15.1398   0.6346
-28.3429   9.8137   4.8278 -28.0177   6.1947  22.0393 -44.7475  13.1565  -3.4911  22.4867
 34.2545  30.0344 -15.9347 -26.9644 -12.2918  -8.0802  30.1956 -11.8246 -32.1063  12.6313
-17.9083  -1.4882  -1.6085   2.6758  -6.2997  11.1905  20.3735  -1.8125   6.3591   9.9902
 16.6749  -7.4828  -3.1152   3.7560 -19.1979  25.6166   6.6457  12.2411  -8.4889  10.2051
-23.6040  15.4773  -4.7529   8.1179 -20.9830  20.9532 -10.5322  24.4085   7.2500  -9.7650
 -1.9036  -4.9139  -6.8719 -43.2947  -0.4436 -30.0724  22.0244 -18.0195   8.6896 -14.4504
 10.9950  13.3954  13.7444   9.6342   8.3571   7.9452  -3.3086  -8.4109  -9.6311  23.1194
  5.1656  -5.2544  27.3565  12.2801  -1.4225  -3.6837 -11.3242  -4.4119 -21.6878  12.9524

第 2 轮 | 设备: cuda:0 | Loss: 308.700317
 48.4466 5